In [ ]:
from dotenv import load_dotenv

load_dotenv(override=True)

In [ ]:
import os

key = os.getenv("OPENAI_API_KEY")

# print(key)

In [ ]:
# from langchain.llms.openai import OpenAI
from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate, ChatPromptTemplate

# llm = OpenAI()
chat = ChatOpenAI(model_name="gpt-4o-mini", temperature=1)

# template, prompt
template = PromptTemplate.from_template("What is the distance between {country_a} and {country_b}? Also, what is your name?")
prompt = template.format(country_a="Mexico", country_b="Thailand")

chat.predict(prompt)

In [ ]:
template = ChatPromptTemplate.from_messages(
    [
       ("system", "You are a geography expert. And you only reply in {language}."),
       ("ai", "Ciao, mi chiamo {name}!"),
       ("human", "What is the distance between {country_a} and {country_b}? Also, what is your name?")
   ]
)

prompt = template.format_messages(
    language="Greek",
    name="Socrates",
    country_a="Mexico",
    country_b="Thailand"
)

chat.predict_messages(prompt)

In [ ]:
from langchain.schema import BaseOutputParser

class CommaOutputParser(BaseOutputParser):
    def parse(self, text):
        items = text.strip().split(",") # 앞뒤 공백 제거 후 "," 기준으로 분리후 리스트로 받는다.

        return list(map(str.strip, items)) # item요소의 순회 해서 앞뒤 공백을 제거한다.

p = CommaOutputParser()
p.parse("Hello, how, are, you")

In [ ]:
template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a list generating machine. Everything you are asked will be answered with a comma seperated list of max {max_items} in lowercase. Do NOT reply with anything else."),
        ("human", "{question}")
    ]
)

In [ ]:
chain = template | chat | CommaOutputParser()

chain.invoke({
    "max_items": 5,
    "question": "What are the pokemons?"
})

In [ ]:
from langchain.prompts import ChatPromptTemplate
from langchain.chat_models import ChatOpenAI
from langchain.callbacks import StreamingStdOutCallbackHandler

chat = ChatOpenAI(temperature=1, streaming=True, callbacks=[StreamingStdOutCallbackHandler()])

chef_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a world-class international chef. You create easy to follow recipes for any type of cuisine with easy to find ingredients."),
    ("human", "I want to cook {cuisine} food.")
])

chef_chain = chef_prompt | chat


In [ ]:
veg_chef_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a vegetarian chef specialized on making traditional recipes vegetarian. You find alternative ingredients and explain their preparation. You don't radically modify the recipe. If there is no alternative for a food just say you don't know how to replace it."
    ),
    (
        "human",
        "{recipe}"
    )
])

veg_chain = veg_chef_prompt | chat

final_chain = {"recipe" : chef_chain} | veg_chain

final_chain.invoke({
    "cuisine" : "indian"
})

## 1. Welcome To Langchain 챌린지

In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.callbacks import StreamingStdOutCallbackHandler

chat = ChatOpenAI(temperature=1, streaming=True, callbacks=[StreamingStdOutCallbackHandler()])

# 프로그래밍 언어에 대한 Haiku를 작성하는 Prompt
haiku_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a poet specialized in writing Haikus about programming languages"),
    ("human", "Write a Haiku about {language}.")
])

# Haiku를 생성하는 Chain
haiku_chain = haiku_prompt | chat

# Haiku를 설명하는 Prompt
explanation_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert specialized in explaining Haikus. Explain the meaning of the Haiku and include the original Haiku in your answer."),
    ("human", "{haiku}")
])

# Haiku를 설명하는 Chain
explanation_chain = explanation_prompt | chat

final_chain = {"haiku" : haiku_chain} | explanation_chain

final_chain.invoke({
    "language" : "Python"
})

## 4-1. FewShotPromptTemplate

In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.prompts.few_shot import FewShotPromptTemplate
from langchain.callbacks import StreamingStdOutCallbackHandler

chat = ChatOpenAI(
    temperature=0.1,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()]
)

template = PromptTemplate(template="What is the capital of {country}", input_variables=["country"])
template.format(country="France")

In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.prompts.few_shot import FewShotPromptTemplate
from langchain.callbacks import StreamingStdOutCallbackHandler

chat = ChatOpenAI(
    temperature=0.1,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()]
)

# 1. 예제를 작성
examples = [
    {
        "question": "What do you know about France?",
        "answer": """
        Here is what I know:
        Capital: Paris
        Language: French
        Food: Wine and Cheese
        Currency: Euro
        """,
    },
    {
        "question": "What do you know about Italy?",
        "answer": """
        I know this:
        Capital: Rome
        Language: Italian
        Food: Pizza and Pasta
        Currency: Euro
        """,
    },
    {
        "question": "What do you know about Greece?",
        "answer": """
        I know this:
        Capital: Athens
        Language: Greek
        Food: Souvlaki and Feta Cheese
        Currency: Euro
        """,
    },
]

# example_tempalte = """
#     Human:{question}
#     AI:{answer}
# """

# PromptTemplate.from_template(example_tempalte)

example_prompt = PromptTemplate.from_template("Human:{question}\nAI:{answer}")

prompt = FewShotPromptTemplate(
    example_prompt=example_prompt, 
    examples=examples, 
    suffix="What do you know about {country}?", 
    input_variables=["country"]
)

prompt.format(country="Germany")

chain = prompt | chat
chain.invoke({
    "country" : "Korea"
})


## 4-2. FewShotChatMessagePromptTemplate

In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.prompts.few_shot import FewShotChatMessagePromptTemplate
from langchain.callbacks import StreamingStdOutCallbackHandler

chat = ChatOpenAI(
    temperature=0.1,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()]
)

# 1. 예제를 작성
examples = [
    {
        "country": "France",
        "answer": """
        Here is what I know:
        Capital: Paris
        Language: French
        Food: Wine and Cheese
        Currency: Euro
        """,
    },
    {
        "country": "Italy",
        "answer": """
        I know this:
        Capital: Rome
        Language: Italian
        Food: Pizza and Pasta
        Currency: Euro
        """,
    },
    {
        "country": "Greece",
        "answer": """
        I know this:
        Capital: Athens
        Language: Greek
        Food: Souvlaki and Feta Cheese
        Currency: Euro
        """,
    },
]


example_prompt = ChatPromptTemplate.from_messages([
    ("human", "What do you know about {country}?"),
    ("ai", "{answer}")
])

example_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt, 
    examples=examples
)

final_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a geography expert"),
    example_prompt,
    ("human", "What do you know about {country}?")
])

chain = final_prompt | chat
chain.invoke({
    "country" : "Germany"
})


## 4-3. LengthBasedExampleSelector

In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.prompts.few_shot import FewShotPromptTemplate
from langchain.callbacks import StreamingStdOutCallbackHandler
from langchain.prompts.example_selector import LengthBasedExampleSelector

chat = ChatOpenAI(
    temperature=0.1,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()]
)

# 1. 예제를 작성
examples = [
    {
        "question": "What do you know about France?",
        "answer": """
        Here is what I know:
        Capital: Paris
        Language: French
        Food: Wine and Cheese
        Currency: Euro
        """,
    },
    {
        "question": "What do you know about Italy?",
        "answer": """
        I know this:
        Capital: Rome
        Language: Italian
        Food: Pizza and Pasta
        Currency: Euro
        """,
    },
    {
        "question": "What do you know about Greece?",
        "answer": """
        I know this:
        Capital: Athens
        Language: Greek
        Food: Souvlaki and Feta Cheese
        Currency: Euro
        """,
    },
]


example_prompt = PromptTemplate.from_template("Human:{question}\nAI:{answer}")

example_selector = LengthBasedExampleSelector(
    examples=examples,
    example_prompt=example_prompt,
    max_length=180, # Few-shot 프롬프트에 포함할 예제들의 최대 길이를 제한한다.
)

prompt = FewShotPromptTemplate(
    example_prompt=example_prompt,
    example_selector=example_selector,
    suffix="What do you know about {country}?", 
    input_variables=["country"]
)

prompt.format(country="Brazil")

## 4-4. Serialization and Composition

In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain.callbacks import StreamingStdOutCallbackHandler
from langchain.prompts import load_prompt

prompt = load_prompt("./prompt.json")

chat = ChatOpenAI(
    temperature=0.1,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()]
)

prompt.format(country="Germany")

In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain.callbacks import StreamingStdOutCallbackHandler
from langchain.prompts import load_prompt

prompt = load_prompt("./prompt.yaml")

chat = ChatOpenAI(
    temperature=0.1,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()]
)

prompt.format(country="Germany")

In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain.callbacks import StreamingStdOutCallbackHandler
from langchain.prompts import PromptTemplate
from langchain.prompts.pipeline import PipelinePromptTemplate # 많은 프롬프트들을 하나로 합칠수 있도록 해준다.

chat = ChatOpenAI(
    temperature=0.1,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()]
)

intro = PromptTemplate.from_template(
    """
    You are a role playing assistant.
    And you are impersonating a {character}
"""
)

example = PromptTemplate.from_template(
    """
    This is an example of how you talk:

    Human: {example_question}
    You: {example_answer}
"""
)

start = PromptTemplate.from_template(
    """
    Start now!

    Human: {question}
    You:
"""
)

final = PromptTemplate.from_template(
    """
    {intro}
                                     
    {example}
                              
    {start}
"""
)

prompts = [
    ("intro", intro),
    ("example", example),
    ("start", start)
]

full_prompt = PipelinePromptTemplate(final_prompt=final, pipeline_prompts=prompts)

full_prompt.format(
    character="Pirate",
    example_question="What is your location?",
    example_answer="Arrrrg! That is a secreat!! Arg arg!!",
    question="What is your fav food?"
)

chain = full_prompt | chat

chain.invoke({
    "character":"Pirate",
    "example_question":"What is your location?",
    "example_answer":"Arrrrg! That is a secreat!! Arg arg!!",
    "question":"What is your fav food?"
})

## 4-5. Caching 

In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain.globals import set_llm_cache, set_debug
from langchain.cache import InMemoryCache

# 이미 답변한 답을 캐싱을 이용하여 저장하여 재사용 가능하다.
set_llm_cache(InMemoryCache())

# 무슨일을 하고 있는지에 대한 로그를 보여준다.
set_debug(True)

chat = ChatOpenAI(
    temperature=0.1,
    # streaming=True,
    # callbacks=[StreamingStdOutCallbackHandler()]
)

chat.predict("How do you make italian pasta.")


In [ ]:
# 즉시 대답 해준다.
chat.predict("How do you make italian pasta.")

## 4-6. Serialization

In [1]:
from langchain.chat_models import ChatOpenAI
from langchain.callbacks import get_openai_callback
# 먼저 한 번 호출 할때 마다 얼마나 돈을 지불하고 있는지 알아보자.

chat = ChatOpenAI(
    temperature=0.1,
)

with get_openai_callback() as usage:
    a = chat.predict("What is the recipe for soju")
    b = chat.predict("What is the recipe for bread")
    print(a, b, "\n")
    print(usage)


Ingredients:
- 1 cup of rice
- 1 cup of water
- 1 tablespoon of nuruk (fermentation starter)
- 1 tablespoon of sugar

Instructions:
1. Rinse the rice thoroughly and soak it in water for at least 1 hour.
2. Drain the rice and steam it until it is fully cooked.
3. Let the rice cool down to room temperature.
4. In a large bowl, mix the nuruk and sugar with the cooked rice.
5. Cover the bowl with a clean cloth and let it ferment for 3-4 days in a warm place.
6. After fermentation, strain the mixture through a cheesecloth to remove any solids.
7. Transfer the liquid to a clean container and let it sit for another 2-3 days to allow the flavors to develop.
8. Your homemade soju is now ready to be enjoyed! Serve chilled or at room temperature. Ingredients:
- 3 1/2 cups all-purpose flour
- 1 packet active dry yeast
- 1 1/2 cups warm water
- 2 tablespoons sugar
- 1 teaspoon salt
- 2 tablespoons olive oil

Instructions:
1. In a large mixing bowl, combine the warm water, sugar, and yeast. Let it s

## 5-1. ConversationBufferMemory

In [3]:
# 첫 번째 메모리는 Conversation Buffer Memory, 그냥 단순하게 이전 대화 내용 전체를 저장하는 메모리이다.
# 이 메모리 단점은 대화 내용이 클 수록 메모리도 계속 커지므로 비효율적이다.
from langchain.memory import ConversationBufferMemory

# memory = ConversationBufferMemory()
memory = ConversationBufferMemory(return_messages=True)

memory.save_context({"input" : "Hi"}, {"output" : "How are you?"})

memory.load_memory_variables({})

# 이 메모리는 언제 사용하면 좋을까?
# text completion 할때 유용하다. 예측 해야할때, 텍스트 자동완성 하고 싶을때.

{'history': [HumanMessage(content='Hi'), AIMessage(content='How are you?')]}

In [5]:
memory.save_context({"input" : "Hi"}, {"output" : "How are you?"})

memory.load_memory_variables({})

{'history': [HumanMessage(content='Hi'),
  AIMessage(content='How are you?'),
  HumanMessage(content='Hi'),
  AIMessage(content='How are you?'),
  HumanMessage(content='Hi'),
  AIMessage(content='How are you?')]}

In [6]:
memory.save_context({"input" : "Hi"}, {"output" : "How are you?"})

memory.load_memory_variables({})

{'history': [HumanMessage(content='Hi'),
  AIMessage(content='How are you?'),
  HumanMessage(content='Hi'),
  AIMessage(content='How are you?'),
  HumanMessage(content='Hi'),
  AIMessage(content='How are you?'),
  HumanMessage(content='Hi'),
  AIMessage(content='How are you?')]}